# Brazilian E-Commerce Dataset Profiling

## Objective

This notebook profiles the raw Olist Brazilian E-Commerce dataset before database loading.

The objectives are to:

- Understand the structure and grain of each dataset
- Validate row and column counts
- Inspect data types
- Identify missing values
- Identify duplicate records
- Examine key fields and relationships
- Establish a baseline for downstream data quality validation

No transformations are performed in this notebook. The raw source data is inspected as provided.

In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")

# Confirm the data folder exists
print(f"\nData folder exists: {DATA_PATH.exists()}")

# List CSV files
csv_files = sorted(DATA_PATH.glob("*.csv"))

print(f"\nCSV files found: {len(csv_files)}")

for file in csv_files:
    print(f"- {file.name}")

Project root: c:\Users\tanya\OneDrive\Desktop\TANYA JOBS\POWER BI\Brazilian-Ecommerce-BI
Data path: c:\Users\tanya\OneDrive\Desktop\TANYA JOBS\POWER BI\Brazilian-Ecommerce-BI\data

Data folder exists: True

CSV files found: 10
- daily-website-visitors.csv
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


## 1. In-Scope Olist Datasets

The project uses the nine datasets that form the Brazilian E-Commerce Public Dataset by Olist.

The `daily-website-visitors.csv` file is present in the local data directory but is outside the scope of the reference project and will not be used in the analysis.

In [2]:
# In-scope Olist datasets

olist_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

print(f"Olist datasets in scope: {len(olist_files)}\n")

for table_name, filename in olist_files.items():
    file_path = DATA_PATH / filename
    print(f"{table_name:<22} → {filename:<45} Exists: {file_path.exists()}")

Olist datasets in scope: 9

customers              → olist_customers_dataset.csv                   Exists: True
geolocation            → olist_geolocation_dataset.csv                 Exists: True
order_items            → olist_order_items_dataset.csv                 Exists: True
order_payments         → olist_order_payments_dataset.csv              Exists: True
order_reviews          → olist_order_reviews_dataset.csv               Exists: True
orders                 → olist_orders_dataset.csv                      Exists: True
products               → olist_products_dataset.csv                    Exists: True
sellers                → olist_sellers_dataset.csv                     Exists: True
category_translation   → product_category_name_translation.csv         Exists: True


## 2. Dataset Structure and Record Counts

This section establishes the basic structure of each in-scope dataset, including:

- Number of rows
- Number of columns
- Column names
- Data types

The purpose is to understand the grain and structure of each source table before loading the data into PostgreSQL.

In [3]:
# Load each Olist dataset and capture basic structural information

datasets = {}

for table_name, filename in olist_files.items():
    file_path = DATA_PATH / filename
    
    df = pd.read_csv(file_path)
    datasets[table_name] = df
    
    print(f"\n{'=' * 70}")
    print(f"Dataset: {table_name}")
    print(f"File: {filename}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print("Columns:")
    
    for column in df.columns:
        print(f"  - {column}: {df[column].dtype}")


Dataset: customers
File: olist_customers_dataset.csv
Rows: 99,441
Columns: 5
Columns:
  - customer_id: str
  - customer_unique_id: str
  - customer_zip_code_prefix: int64
  - customer_city: str
  - customer_state: str

Dataset: geolocation
File: olist_geolocation_dataset.csv
Rows: 1,000,163
Columns: 5
Columns:
  - geolocation_zip_code_prefix: int64
  - geolocation_lat: float64
  - geolocation_lng: float64
  - geolocation_city: str
  - geolocation_state: str

Dataset: order_items
File: olist_order_items_dataset.csv
Rows: 112,650
Columns: 7
Columns:
  - order_id: str
  - order_item_id: int64
  - product_id: str
  - seller_id: str
  - shipping_limit_date: str
  - price: float64
  - freight_value: float64

Dataset: order_payments
File: olist_order_payments_dataset.csv
Rows: 103,886
Columns: 5
Columns:
  - order_id: str
  - payment_sequential: int64
  - payment_type: str
  - payment_installments: int64
  - payment_value: float64

Dataset: order_reviews
File: olist_order_reviews_dataset.csv

## 3. Key Uniqueness and Table Grain

This section examines candidate key fields to determine the actual grain of each dataset.

The analysis focuses on:

- Number of distinct values
- Number of duplicate values
- Maximum records associated with a key
- Potential one-to-many relationships

The results will be used to design the PostgreSQL schema and prevent double-counting during SQL and Power BI analysis.

In [4]:
# Examine candidate keys and their uniqueness

key_checks = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"]
}

for table_name, columns in key_checks.items():
    df = datasets[table_name]
    
    print(f"\n{'=' * 70}")
    print(f"Dataset: {table_name}")
    
    for column in columns:
        distinct_count = df[column].nunique(dropna=False)
        duplicate_rows = len(df) - distinct_count
        
        print(
            f"{column:<35} "
            f"Distinct: {distinct_count:>10,} | "
            f"Duplicate rows: {duplicate_rows:>10,}"
        )


Dataset: customers
customer_id                         Distinct:     99,441 | Duplicate rows:          0
customer_unique_id                  Distinct:     96,096 | Duplicate rows:      3,345

Dataset: orders
order_id                            Distinct:     99,441 | Duplicate rows:          0

Dataset: order_items
order_id                            Distinct:     98,666 | Duplicate rows:     13,984
order_item_id                       Distinct:         21 | Duplicate rows:    112,629
product_id                          Distinct:     32,951 | Duplicate rows:     79,699
seller_id                           Distinct:      3,095 | Duplicate rows:    109,555

Dataset: order_payments
order_id                            Distinct:     99,440 | Duplicate rows:      4,446
payment_sequential                  Distinct:         29 | Duplicate rows:    103,857

Dataset: order_reviews
review_id                           Distinct:     98,410 | Duplicate rows:        814
order_id                        

## 4. Dataset Grain and Key Findings

| Dataset | Grain | Candidate Key | Important Relationship |
|---|---|---|---|
| Customers | One customer record | `customer_id` | Multiple `customer_id` values can belong to one `customer_unique_id` |
| Orders | One order | `order_id` | One customer can have multiple orders |
| Order Items | One item within an order | (`order_id`, `order_item_id`) | One order can contain multiple items |
| Order Payments | One payment record for an order | (`order_id`, `payment_sequential`) | One order can have multiple payment records |
| Order Reviews | One review record associated with an order | Requires investigation | An order may have multiple review records |
| Products | One product | `product_id` | Products appear across order items |
| Sellers | One seller | `seller_id` | Sellers appear across order items |
| Geolocation | One geolocation record | Not simply ZIP prefix | Multiple records can exist for one ZIP prefix |
| Category Translation | One category translation | `product_category_name` | Lookup table for English category names |

### Key analytical implications

- `customer_unique_id` should be used when measuring unique customers across customer records.
- `order_item_id` is not globally unique and should not be used as a standalone primary key.
- `payment_sequential` is not globally unique and should not be used as a standalone primary key.
- Geolocation must not be joined as a one-to-one ZIP-prefix table without further validation.
- Review duplication requires investigation before any cleaning decision is made.

## 5. Review Duplicate Investigation

Repeated `review_id` and `order_id` values were identified during key profiling.

These records will be investigated before deciding whether they represent:
- legitimate multiple review records,
- repeated review records,
- or data-quality issues.

No records will be removed during profiling.

In [5]:
# Investigate repeated review IDs

reviews = datasets["order_reviews"]

duplicate_review_ids = (
    reviews[reviews["review_id"].duplicated(keep=False)]
    .sort_values(["review_id", "order_id"])
)

print(f"Rows involved in repeated review IDs: {len(duplicate_review_ids):,}")
print(f"Repeated review IDs: {duplicate_review_ids['review_id'].nunique():,}")

duplicate_review_ids.head(20)

Rows involved in repeated review IDs: 1,603
Repeated review IDs: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


## 6. Review Grain Investigation

The initial profiling identified repeated `review_id` values.

Inspection shows that some repeated review IDs are associated with different orders while retaining identical review attributes.

This section investigates the relationship between review IDs and orders to determine the appropriate analytical grain.

No records are removed at this stage.

In [6]:
# Investigate how many orders are associated with each review_id

review_id_order_counts = (
    reviews.groupby("review_id")["order_id"]
    .nunique()
    .reset_index(name="order_count")
)

print("Review IDs associated with multiple orders:")
print(
    review_id_order_counts[
        review_id_order_counts["order_count"] > 1
    ]
    .sort_values("order_count", ascending=False)
    .head(20)
)

print(
    f"\nTotal review IDs associated with multiple orders: "
    f"{(review_id_order_counts['order_count'] > 1).sum():,}"
)

Review IDs associated with multiple orders:
                              review_id  order_count
25390  4219a80ab469e3fc9901437b73da3f75            3
47465  7b606b0d57b078384f0b58eac1d41d78            3
4762   0c76e7a547a531e7bf9f0b99cba071c1            3
40583  69a1068c3128a14994e3e422e4539e04            3
18600  308316408775d1600dad81bd3184556d            3
26458  44e9f871226d8a130de3fc39dfbdf0c5            3
12796  2172867fd5b1a55f98fe4608e1547b4b            3
60829  9e25d6e3025e9b9a0fc7f03588d33e2b            3
21687  38821b5c496b678cf91acc34892805ad            3
43169  70509c441d994fa03d6c1457930c9024            3
19228  32415bbf6e341d5d517080a796f79b5c            3
19966  3415c9f764e478409e8e0660ae816dd2            3
29612  4d0e6dd087008d1f992d25ef6e1f619f            3
26600  4548534449b1f572e357211b90724f1b            3
17423  2d6ac45f859465b5c185274a1c929637            3
22147  39b4603793c1c7f5f36d809b4a218664            3
12172  1fb4ddc969e6bea80e38deec00393a6f            3
31

## 7. Review Record Duplication Check

This check determines whether repeated review IDs represent exact duplicate records or records associated with different orders.

The raw source data will be preserved regardless of the result.

In [7]:
# Check for exact duplicate review records

exact_duplicate_reviews = reviews[
    reviews.duplicated(keep=False)
]

print(
    f"Rows that are exact duplicates across all review columns: "
    f"{len(exact_duplicate_reviews):,}"
)

print(
    f"Number of exact duplicate row groups: "
    f"{exact_duplicate_reviews.duplicated().sum():,}"
)

exact_duplicate_reviews.head(20)

Rows that are exact duplicates across all review columns: 0
Number of exact duplicate row groups: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


## 8. Missing-Value Profiling

This section evaluates missing values across all in-scope Olist datasets.

Missing values are not automatically treated as data-quality errors. Their significance depends on the business meaning and whether the field is expected to be populated for every record.

The results will be used to distinguish:
- expected missing values,
- conditionally missing values,
- and potential data-quality issues.

In [8]:
# Profile missing values across all Olist datasets

for table_name, df in datasets.items():
    
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    missing_summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": missing_pct
    })
    
    missing_summary = missing_summary[
        missing_summary["missing_count"] > 0
    ].sort_values("missing_count", ascending=False)
    
    print(f"\n{'=' * 70}")
    print(f"Dataset: {table_name}")
    
    if missing_summary.empty:
        print("No missing values found.")
    else:
        print(missing_summary.to_string())


Dataset: customers
No missing values found.

Dataset: geolocation
No missing values found.

Dataset: order_items
No missing values found.

Dataset: order_payments
No missing values found.

Dataset: order_reviews
                        missing_count  missing_pct
review_comment_title            87656        88.34
review_comment_message          58247        58.70

Dataset: orders
                               missing_count  missing_pct
order_delivered_customer_date           2965         2.98
order_delivered_carrier_date            1783         1.79
order_approved_at                        160         0.16

Dataset: products
                            missing_count  missing_pct
product_category_name                 610         1.85
product_name_lenght                   610         1.85
product_description_lenght            610         1.85
product_photos_qty                    610         1.85
product_weight_g                        2         0.01
product_length_cm                   

## 9. Order Date Missingness vs Order Status

Missing delivery and approval timestamps are evaluated against `order_status` to determine whether the missing values are consistent with the order lifecycle.

The objective is to distinguish expected process-state missingness from potential data-quality issues.

In [9]:
# Investigate missing order timestamps by order status

orders = datasets["orders"]

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in date_columns:
    print(f"\n{'=' * 70}")
    print(f"Missing values in: {column}")
    
    result = (
        orders[orders[column].isna()]
        .groupby("order_status")
        .size()
        .sort_values(ascending=False)
    )
    
    print(result)


Missing values in: order_approved_at
order_status
canceled     141
delivered     14
created        5
dtype: int64

Missing values in: order_delivered_carrier_date
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
dtype: int64

Missing values in: order_delivered_customer_date
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
dtype: int64


## 10. Delivered Orders Missing Customer Delivery Date

Delivered orders should normally contain a customer delivery timestamp.

The eight records identified during missing-value profiling are inspected individually to determine whether they represent source-data anomalies or another legitimate condition.

In [10]:
# Inspect delivered orders missing the customer delivery timestamp

delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

print(f"Records found: {len(delivered_missing_date)}")

delivered_missing_date

Records found: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


## 11. Additional Delivered-Order Timestamp Exceptions

Additional validation is performed for delivered orders missing approval or carrier timestamps.

In [11]:
# Inspect delivered orders with missing approval or carrier timestamps

delivered_exceptions = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna()
    )
]

print(f"Delivered orders with timestamp exceptions: {len(delivered_exceptions)}")

delivered_exceptions[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Delivered orders with timestamp exceptions: 16


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


## 12. Product Missing-Value Investigation

The product dataset contains 610 records with missing values across:

- `product_category_name`
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`

This section determines whether these missing values occur on the same product records and whether the records have other identifying information available.

In [12]:
# Investigate products with missing category and related attributes

products = datasets["products"]

product_missing_group = products[
    products[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty"
        ]
    ].isna().all(axis=1)
]

print(f"Products missing all four fields: {len(product_missing_group)}")

product_missing_group.head(20)

Products missing all four fields: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


## 13. Incomplete Products in Transaction Data

The 610 products with missing category and descriptive attributes are checked against the `order_items` dataset.

The purpose is to determine whether these products appear in actual transactions and therefore affect downstream sales and product-category analysis.

In [13]:
# Check whether products with missing descriptive attributes appear in order items

order_items = datasets["order_items"]

incomplete_product_ids = set(product_missing_group["product_id"])

sold_incomplete_products = order_items[
    order_items["product_id"].isin(incomplete_product_ids)
]

unique_sold_incomplete_products = sold_incomplete_products["product_id"].nunique()

print(f"Incomplete products: {len(incomplete_product_ids):,}")
print(f"Incomplete products appearing in order items: {unique_sold_incomplete_products:,}")
print(f"Order-item records involving incomplete products: {len(sold_incomplete_products):,}")

print("\nTop incomplete products by number of order items:")
print(
    sold_incomplete_products["product_id"]
    .value_counts()
    .head(20)
)

Incomplete products: 610
Incomplete products appearing in order items: 610
Order-item records involving incomplete products: 1,603

Top incomplete products by number of order items:
product_id
5a848e4ab52fd5445cdc07aab1c40e48    197
b1d207586fca400a2370d50a9ba1da98     48
76d1a1a9d21ab677a61c3ae34b1b352f     32
ad88641611c35ebd59ecda07a9f17099     29
3b60d513e90300a4e9833e5cda1f1d61     29
4914f8796af2ecd359fd8f44b9b92339     28
0502d1a36be75bd36b452f31c6ed264a     26
e0f33a3329af6716a0bb47fd7a664439     24
b36f3c918c91478c4559160022d3f14e     17
c230b471b7e21ff9060e68ee154afd70     17
5eb564652db742ff8f28759cd8d2652a     17
1b0e39ec889889ea1d492603d8512bfb     15
4e0d588f8e002f2bad9cbe0b8f66f6f6     15
17823ffd2de8234f0e885a71109613a4     11
e891d4a9622cae3b9fc2ec558bda155b     11
1b7384e0f9f5e4cb914cd3f5535a4cab     10
6ab9d49c399f239bc88ea28c7568213f     10
3d4784f9a07ca5de370c86800589c860     10
f9b1795281ce51b1cf39ef6d101ae8ab     10
a4d8f727f92014da5dd64116af14634c     10
Name: c

## 14. Product Physical Attribute Missingness

The product dataset contains two records with missing physical attributes.

These records are investigated to determine:
- whether the missing fields occur on the same products,
- whether the products appear in transactions,
- and whether the missing values affect freight or logistics analysis.

In [14]:
# Investigate products with missing physical attributes

physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

physical_missing = products[
    products[physical_columns].isna().any(axis=1)
]

print(f"Products with at least one missing physical attribute: {len(physical_missing)}")

physical_missing

Products with at least one missing physical attribute: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 15. Incomplete Physical Attributes in Transaction Data

Products with missing physical attributes are checked against the order-item dataset to determine whether the missing information affects actual transactions and downstream logistics analysis.

In [15]:
# Check whether products with missing physical attributes appear in transactions

physical_product_ids = set(physical_missing["product_id"])

sold_physical_missing = order_items[
    order_items["product_id"].isin(physical_product_ids)
]

print(f"Products with missing physical attributes: {len(physical_product_ids):,}")
print(
    f"Products appearing in order items: "
    f"{sold_physical_missing['product_id'].nunique():,}"
)
print(
    f"Order-item records involving these products: "
    f"{len(sold_physical_missing):,}"
)

print("\nOrder-item counts by product:")
print(
    sold_physical_missing["product_id"]
    .value_counts()
)

Products with missing physical attributes: 2
Products appearing in order items: 2
Order-item records involving these products: 18

Order-item counts by product:
product_id
5eb564652db742ff8f28759cd8d2652a    17
09ff539a621711667c43eba6a3bd8466     1
Name: count, dtype: int64


## 16. Numeric Data Validation

This section checks numeric fields for values that may violate expected business or data constraints.

The checks focus on:

- Negative monetary values
- Invalid payment values
- Invalid installment counts
- Review scores outside the expected range
- Invalid product quantities
- Non-positive physical dimensions

Potential exceptions will be investigated before any treatment or transformation is applied.

In [16]:
# Numeric validation checks

print("=" * 70)
print("ORDER ITEMS")
print("=" * 70)

print("\nNegative price values:")
print((order_items["price"] < 0).sum())

print("\nNegative freight values:")
print((order_items["freight_value"] < 0).sum())


print("\n" + "=" * 70)
print("ORDER PAYMENTS")
print("=" * 70)

print("\nNegative payment values:")
print((datasets["order_payments"]["payment_value"] < 0).sum())

print("\nPayment installments <= 0:")
print((datasets["order_payments"]["payment_installments"] <= 0).sum())


print("\n" + "=" * 70)
print("ORDER REVIEWS")
print("=" * 70)

print("\nReview scores outside 1-5:")
print(
    (
        (reviews["review_score"] < 1) |
        (reviews["review_score"] > 5)
    ).sum()
)


print("\n" + "=" * 70)
print("PRODUCTS")
print("=" * 70)

product_numeric_checks = {
    "product_name_lenght < 0": products["product_name_lenght"] < 0,
    "product_description_lenght < 0": products["product_description_lenght"] < 0,
    "product_photos_qty < 0": products["product_photos_qty"] < 0,
    "product_weight_g <= 0": products["product_weight_g"] <= 0,
    "product_length_cm <= 0": products["product_length_cm"] <= 0,
    "product_height_cm <= 0": products["product_height_cm"] <= 0,
    "product_width_cm <= 0": products["product_width_cm"] <= 0,
}

for check_name, condition in product_numeric_checks.items():
    print(f"{check_name}: {condition.sum()}")

ORDER ITEMS

Negative price values:
0

Negative freight values:
0

ORDER PAYMENTS

Negative payment values:
0

Payment installments <= 0:
2

ORDER REVIEWS

Review scores outside 1-5:
0

PRODUCTS
product_name_lenght < 0: 0
product_description_lenght < 0: 0
product_photos_qty < 0: 0
product_weight_g <= 0: 4
product_length_cm <= 0: 0
product_height_cm <= 0: 0
product_width_cm <= 0: 0


## 17. Payment Installment Validation

Two payment records contain `payment_installments <= 0`.

These records are inspected to determine whether they represent invalid source values or a specific payment condition.

In [17]:
# Inspect payment records with non-positive installments

payments = datasets["order_payments"]

invalid_installments = payments[
    payments["payment_installments"] <= 0
]

print(f"Records found: {len(invalid_installments)}")

invalid_installments

Records found: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


## 18. Product Weight Validation

Four products contain non-positive values for `product_weight_g`.

These records are inspected to determine whether the values are zero or negative and whether the affected products appear in transaction data.

In [18]:
# Inspect products with non-positive weight

invalid_weights = products[
    products["product_weight_g"] <= 0
]

print(f"Records found: {len(invalid_weights)}")

invalid_weights

Records found: 4


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


## 19. Zero-Weight Products in Transaction Data

The four products with `product_weight_g = 0` are checked against the order-item dataset to determine whether the invalid weight values affect actual transactions.

In [19]:
# Check zero-weight products in transactions

zero_weight_product_ids = set(invalid_weights["product_id"])

sold_zero_weight = order_items[
    order_items["product_id"].isin(zero_weight_product_ids)
]

print(f"Zero-weight products: {len(zero_weight_product_ids):,}")
print(
    f"Zero-weight products appearing in order items: "
    f"{sold_zero_weight['product_id'].nunique():,}"
)
print(
    f"Order-item records involving zero-weight products: "
    f"{len(sold_zero_weight):,}"
)

print("\nOrder-item counts by product:")
print(sold_zero_weight["product_id"].value_counts())

Zero-weight products: 4
Zero-weight products appearing in order items: 4
Order-item records involving zero-weight products: 8

Order-item counts by product:
product_id
e673e90efa65a5409ff4196c038bb5af    4
36ba42dd187055e1fbe943b2d11430ca    2
81781c0fed9fe1ad6e8c81fca1e1cb08    1
8038040ee2a71048d4bdbbdc985b69ab    1
Name: count, dtype: int64


## 20. Date and Timestamp Validation

This section validates chronological relationships between timestamp fields.

The objective is to identify records where the sequence of order events may violate the expected order lifecycle.

The validation checks for:
- Approval occurring before purchase
- Carrier handoff occurring before purchase or approval
- Customer delivery occurring before purchase or carrier handoff
- Review response occurring before review creation
- Invalid or contradictory timestamp sequences

Records with missing timestamps are evaluated only when both timestamps required for a comparison are available.

In [20]:
# Convert order timestamp columns to datetime for validation

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

print("Order timestamp columns converted to datetime.")
print(orders[order_date_columns].dtypes)

Order timestamp columns converted to datetime.
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## 21. Order Timestamp Sequence Validation

The order lifecycle is expected to follow this general sequence:

Purchase → Approval → Carrier Handoff → Customer Delivery

Each chronological relationship is tested only when both timestamps required for the comparison are available.

In [21]:
# Validate chronological order of order lifecycle timestamps

checks = {
    "Approval before purchase": (
        orders["order_approved_at"].notna() &
        orders["order_purchase_timestamp"].notna() &
        (orders["order_approved_at"] < orders["order_purchase_timestamp"])
    ),
    
    "Carrier handoff before purchase": (
        orders["order_delivered_carrier_date"].notna() &
        orders["order_purchase_timestamp"].notna() &
        (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"])
    ),
    
    "Carrier handoff before approval": (
        orders["order_delivered_carrier_date"].notna() &
        orders["order_approved_at"].notna() &
        (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
    ),
    
    "Customer delivery before purchase": (
        orders["order_delivered_customer_date"].notna() &
        orders["order_purchase_timestamp"].notna() &
        (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"])
    ),
    
    "Customer delivery before carrier handoff": (
        orders["order_delivered_customer_date"].notna() &
        orders["order_delivered_carrier_date"].notna() &
        (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
    ),
}

for check_name, condition in checks.items():
    print(f"{check_name}: {condition.sum():,}")

Approval before purchase: 0
Carrier handoff before purchase: 166
Carrier handoff before approval: 1,359
Customer delivery before purchase: 0
Customer delivery before carrier handoff: 23


## 22. Customer Delivery Before Carrier Handoff

Twenty-three orders were identified where the recorded customer delivery timestamp occurs before the recorded carrier handoff timestamp.

These records are inspected individually to determine whether the timestamps represent source-data inconsistencies or another operational interpretation.

In [22]:
# Inspect orders where customer delivery precedes carrier handoff

customer_before_carrier = orders[
    orders["order_delivered_customer_date"].notna() &
    orders["order_delivered_carrier_date"].notna() &
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    )
]

print(f"Records found: {len(customer_before_carrier)}")

customer_before_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Records found: 23


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18
